# ARIMAX — UK Fruit & Veg Price Forecasting
**Author:** Dani  
**Input:** `data/merged_features_weekly.csv` (20 commodities, weekly)  
**Model:** `ARIMAX(1,1,1)` — ARIMA with exogenous regressors (xreg)  

**Exogenous regressors used:**
- `api_energy_and_lubricants`, `api_fertilisers_and_soil_improvers`,
  `api_plant_protection_products`, `api_fresh_fruit`, `api_fresh_vegetables`
- `fuel_petrol_price`, `fuel_diesel_price`, `sppi_road_freight`
- `shock_2021q4_2023q1`, `post_shock` (structural break dummies)
- `week_sin`, `week_cos` (harmonic seasonality terms)

**Outputs** (written relative to project root):
- `outputs/dani_models/arimax/arimax_metrics.csv`
- `outputs/dani_models/arimax/arimax_residual_diagnostics.csv`
- `outputs/dani_models/arimax/arimax_coefficients.csv`
- `outputs/dani_models/arimax/forecasts/<commodity>.csv`
- `outputs/dani_models/arimax/plots/<commodity>_forecast.png`
- `outputs/dani_models/arimax/plots/<commodity>_residuals.png`
- `outputs/dani_models/arimax/plots/<commodity>_coefs.png`

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Path resolution ────────────────────────────────────────────────────────
NB_DIR = Path().resolve()

def find_project_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'data' / 'merged_features_weekly.csv').exists():
            return parent
    raise FileNotFoundError(
        f'Cannot locate project root starting from {start}'
    )

ROOT = find_project_root(NB_DIR)
print(f'Project root : {ROOT}')

INPUT_CSV    = ROOT / 'data' / 'merged_features_weekly.csv'
OUTPUT_DIR   = ROOT / 'outputs' / 'dani_models' / 'arimax'
FORECAST_DIR = OUTPUT_DIR / 'forecasts'
PLOT_DIR     = OUTPUT_DIR / 'plots'

for d in (FORECAST_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Input CSV    : {INPUT_CSV}')
print(f'Output dir   : {OUTPUT_DIR}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
ALPHA     = 0.05
MIN_TRAIN = 110

# ARIMAX orders — (1,1,1) with no seasonal component
P, D, Q = 1, 1, 1

# Exogenous regressors — all candidates; only those present in the CSV are used
EXOG_CANDIDATES = [
    'api_energy_and_lubricants',
    'api_fertilisers_and_soil_improvers',
    'api_plant_protection_products',
    'api_fresh_fruit',
    'api_fresh_vegetables',
    'fuel_petrol_price',
    'fuel_diesel_price',
    'sppi_road_freight',
    'shock_2021q4_2023q1',
    'post_shock',
    'week_sin',
    'week_cos',
]

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV, parse_dates=['date'])
df = df.sort_values(['commodity', 'date']).reset_index(drop=True)

commodities = sorted(df['commodity'].unique())
exog_cols   = [c for c in EXOG_CANDIDATES if c in df.columns]

print(f'Loaded {len(df):,} rows | {len(commodities)} commodities')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Exog columns available ({len(exog_cols)}): {exog_cols}')

if not exog_cols:
    raise ValueError('No exogenous columns found — check EXOG_CANDIDATES against CSV columns')

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

def sanitize(name: str) -> str:
    keep = []
    for ch in name:
        if ch.isalnum() or ch in ('-', '_'):
            keep.append(ch)
        elif ch in (' ', '/', '(', ')'):
            keep.append('_')
    clean = ''.join(keep).strip('_')
    while '__' in clean:
        clean = clean.replace('__', '_')
    return clean or 'unknown'


def train_test_split(g: pd.DataFrame):
    if 'split' in g.columns:
        tr = g[g['split'] == 'train'].copy()
        te = g[g['split'] == 'test'].copy()
        if len(tr) > 0 and len(te) > 0:
            return tr, te
    n = len(g)
    test_n = min(52, max(1, n // 5))
    return g.iloc[:-test_n].copy(), g.iloc[-test_n:].copy()


# ── Metrics ────────────────────────────────────────────────────────────────

def rmse(a, p):   return float(np.sqrt(np.mean((a - p) ** 2)))
def mae(a, p):    return float(np.mean(np.abs(a - p)))
def mape(a, p):
    mask = a != 0
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100) if mask.any() else np.nan
def smape(a, p):
    d = np.abs(a) + np.abs(p)
    mask = d != 0
    return float(np.mean(200 * np.abs(a[mask] - p[mask]) / d[mask])) if mask.any() else np.nan
def mase(a, p, train):
    denom = np.mean(np.abs(np.diff(train)))
    return float(np.mean(np.abs(a - p)) / denom) if denom > 0 else np.nan


def compute_metrics(actual, pred, train, fit, commodity, model_label, n_exog):
    a = np.asarray(actual, dtype=float)
    p = np.asarray(pred,   dtype=float)
    t = np.asarray(train,  dtype=float)
    return {
        'commodity':  commodity,
        'model':      model_label,
        'n_exog':     n_exog,
        'exog_cols':  '|'.join(exog_cols),
        'RMSE':       round(rmse(a, p),    4),
        'MAE':        round(mae(a, p),     4),
        'MASE':       round(mase(a, p, t), 4),
        'sMAPE':      round(smape(a, p),   4),
        'MAPE':       round(mape(a, p),    4),
        'AIC':        round(fit.aic, 2)    if fit.aic is not None else np.nan,
        'BIC':        round(fit.bic, 2)    if fit.bic is not None else np.nan,
        'train_rows': len(t),
        'test_rows':  len(a),
    }


# ── Residual diagnostics ───────────────────────────────────────────────────

def residual_diagnostics(fit, commodity, model_label):
    resid = np.asarray(fit.resid, dtype=float)
    resid = resid[~np.isnan(resid)]

    lb10 = acorr_ljungbox(resid, lags=[10], return_df=True)
    lb20 = acorr_ljungbox(resid, lags=[20], return_df=True)

    if len(resid) <= 5000:
        norm_stat, norm_p = stats.shapiro(resid[:5000])
        norm_name = 'Shapiro-Wilk'
    else:
        norm_stat, norm_p = stats.normaltest(resid)
        norm_name = 'DAgostino-Pearson'

    lb10_p = float(lb10['lb_pvalue'].iloc[0])
    lb20_p = float(lb20['lb_pvalue'].iloc[0])

    return {
        'commodity':         commodity,
        'model':             model_label,
        'resid_mean':        round(float(np.mean(resid)), 6),
        'resid_sd':          round(float(np.std(resid)),  4),
        'ljung_box_lag10_p': round(lb10_p, 4),
        'ljung_box_lag20_p': round(lb20_p, 4),
        'normality_test':    norm_name,
        'normality_p':       round(float(norm_p), 4),
        'autocorr_ok':       lb10_p > ALPHA,
        'normality_ok':      float(norm_p) > ALPHA,
    }


# ── Extract coefficient table ──────────────────────────────────────────────

def extract_coefficients(fit, commodity, model_label):
    summary = fit.summary2().tables[1]
    rows = []
    for term in summary.index:
        row = summary.loc[term]
        rows.append({
            'commodity': commodity,
            'model':     model_label,
            'term':      term,
            'estimate':  float(row['Coef.']),
            'std_err':   float(row['Std.Err.']),
            't_stat':    float(row['t']),
            'p_value':   float(row['P>|t|']),
            'conf_lo':   float(row['[0.025']),
            'conf_hi':   float(row['0.975]']),
            'significant': float(row['P>|t|']) < ALPHA,
        })
    return rows

In [ ]:
# ── Plotting functions ─────────────────────────────────────────────────────

def plot_forecast(train_df, test_df, pred_df, commodity, safe, output_dir):
    fig, ax = plt.subplots(figsize=(13, 5))
    train_plot = train_df[train_df['date'] >= train_df['date'].max() - pd.Timedelta(days=730)]

    ax.plot(train_plot['date'], train_plot['target_price'],
            color='#95a5a6', linewidth=0.8, label='Train')
    ax.plot(test_df['date'], test_df['target_price'],
            color='#2c3e50', linewidth=1.2, label='Actual')
    ax.plot(test_df['date'], pred_df['mean'],
            color='#e74c3c', linewidth=1.2, linestyle='--', label='Forecast')

    if 'lower_95' in pred_df.columns:
        ax.fill_between(test_df['date'], pred_df['lower_95'], pred_df['upper_95'],
                        alpha=0.15, color='#e74c3c', label='95% PI')
    if 'lower_80' in pred_df.columns:
        ax.fill_between(test_df['date'], pred_df['lower_80'], pred_df['upper_80'],
                        alpha=0.25, color='#e74c3c', label='80% PI')

    ax.axvline(test_df['date'].iloc[0], color='grey', linestyle=':', linewidth=0.8)
    ax.set_title(f'ARIMAX(1,1,1)+xreg Forecast — {commodity}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Price (£/unit)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(output_dir / f'{safe}_forecast.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def plot_residuals(fit, commodity, safe, output_dir):
    resid = np.asarray(fit.resid, dtype=float)
    fitted_v = np.asarray(fit.fittedvalues, dtype=float)
    resid_c = resid[~np.isnan(resid)]

    fig = plt.figure(figsize=(14, 11))
    fig.suptitle(f'Residual Diagnostics — ARIMAX — {commodity}',
                 fontsize=13, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(resid, color='#2c3e50', linewidth=0.5)
    ax1.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax1.set_title('Residuals over time')
    ax1.set_xlabel('Week index')
    ax1.set_ylabel('Residual')

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(resid_c, bins=40, density=True, color='#3498db', alpha=0.7, edgecolor='white')
    xr = np.linspace(resid_c.min(), resid_c.max(), 200)
    ax2.plot(xr, stats.norm.pdf(xr, resid_c.mean(), resid_c.std()), color='red', linewidth=1.2)
    ax2.set_title('Residual distribution')
    ax2.set_xlabel('Residual')
    ax2.set_ylabel('Density')

    ax3 = fig.add_subplot(gs[1, 0])
    from statsmodels.graphics.tsaplots import plot_acf
    plot_acf(resid_c, lags=52, ax=ax3, color='#2980b9', zero=False)
    ax3.set_title('ACF of residuals')

    ax4 = fig.add_subplot(gs[1, 1])
    from statsmodels.graphics.tsaplots import plot_pacf
    plot_pacf(resid_c, lags=26, ax=ax4, color='#8e44ad', zero=False)
    ax4.set_title('PACF of residuals')

    ax5 = fig.add_subplot(gs[2, 0])
    (osm, osr), (slope, intercept, r) = stats.probplot(resid_c)
    ax5.scatter(osm, osr, alpha=0.4, s=8, color='#2c3e50')
    ax5.plot(osm, slope * np.array(osm) + intercept, color='red', linewidth=1)
    ax5.set_title('Q-Q plot')
    ax5.set_xlabel('Theoretical quantiles')
    ax5.set_ylabel('Sample quantiles')

    ax6 = fig.add_subplot(gs[2, 1])
    valid = ~np.isnan(resid) & ~np.isnan(fitted_v)
    ax6.scatter(fitted_v[valid], resid[valid], alpha=0.35, s=8, color='#16a085')
    ax6.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax6.set_title('Residuals vs Fitted')
    ax6.set_xlabel('Fitted values')
    ax6.set_ylabel('Residual')

    fig.savefig(output_dir / f'{safe}_residuals.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved residual plot: {safe}')


def plot_coefficients(coef_rows, commodity, safe, output_dir):
    """Forest plot of regression (xreg) coefficients only."""
    reg = [r for r in coef_rows if not any(
        r['term'].startswith(p) for p in ['ar.', 'ma.', 'sigma', 'const', 'intercept']
    )]
    if not reg:
        return

    terms  = [r['term']      for r in reg]
    est    = [r['estimate']  for r in reg]
    lo     = [r['conf_lo']   for r in reg]
    hi     = [r['conf_hi']   for r in reg]
    sig    = [r['significant'] for r in reg]

    order  = np.argsort(est)
    terms  = [terms[i] for i in order]
    est    = [est[i]   for i in order]
    lo     = [lo[i]    for i in order]
    hi     = [hi[i]    for i in order]
    sig    = [sig[i]   for i in order]

    colors = ['#27ae60' if s else '#e74c3c' for s in sig]
    ys     = range(len(terms))

    fig, ax = plt.subplots(figsize=(9, max(4, len(terms) * 0.55)))
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    for i, (y, e, l, h, c) in enumerate(zip(ys, est, lo, hi, colors)):
        ax.plot([l, h], [y, y], color=c, linewidth=1.5)
        ax.scatter([e], [y], color=c, s=40, zorder=3)
    ax.set_yticks(list(ys))
    ax.set_yticklabels(terms, fontsize=9)
    ax.set_xlabel('Coefficient estimate')
    ax.set_title(
        f'ARIMAX Regression Coefficients — {commodity}\n'
        f'Green = p<{ALPHA}  |  Error bars = 95% CI',
        fontsize=11, fontweight='bold'
    )
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig.savefig(output_dir / f'{safe}_coefs.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved coefficient plot: {safe}')

In [ ]:
# ── Main loop: fit ARIMAX for every commodity ──────────────────────────────
MODEL_LABEL = f'ARIMAX({P},{D},{Q})+xreg'

metrics_rows = []
diag_rows    = []
coef_rows    = []
status_rows  = []

for commodity in commodities:
    print(f'\n{"="*60}')
    print(f'  Commodity: {commodity}')
    print(f'{"="*60}')

    safe = sanitize(commodity)
    g = df[df['commodity'] == commodity].sort_values('date').copy()

    train_df, test_df = train_test_split(g)

    print(f'  Train: {train_df["date"].min().date()} → {train_df["date"].max().date()} ({len(train_df)} weeks)')
    print(f'  Test:  {test_df["date"].min().date()} → {test_df["date"].max().date()} ({len(test_df)} weeks)')

    if len(train_df) < MIN_TRAIN:
        msg = f'Skipping: only {len(train_df)} train rows (need {MIN_TRAIN})'
        print(f'  !! {msg}')
        status_rows.append({'commodity': commodity, 'status': 'skipped', 'reason': msg})
        continue

    y_train = pd.to_numeric(train_df['target_price'], errors='coerce').astype(float)
    y_test  = pd.to_numeric(test_df['target_price'],  errors='coerce').astype(float)

    if y_train.isna().any() or y_test.isna().any():
        msg = 'Skipping: NaN in target_price'
        print(f'  !! {msg}')
        status_rows.append({'commodity': commodity, 'status': 'skipped', 'reason': msg})
        continue

    # Prepare exogenous matrices
    # Training: forward/backward fill any internal gaps
    x_train = train_df[exog_cols].apply(pd.to_numeric, errors='coerce').ffill().bfill()
    # Test: some exog columns (API indices, fuel prices) may be NaN for the most recent
    # test period because the data source hasn't been updated yet.
    # Carry forward the last observed training value as the best available forecast.
    x_test  = test_df[exog_cols].apply(pd.to_numeric, errors='coerce')
    last_train_vals = x_train.iloc[-1]          # last row of training exog
    x_test  = x_test.fillna(last_train_vals).bfill().ffill()
    # Reset indices so statsmodels aligns endog and exog correctly
    y_train = y_train.reset_index(drop=True)
    x_train = x_train.reset_index(drop=True)
    x_test  = x_test.reset_index(drop=True)

    if x_train.isna().any().any() or x_test.isna().any().any():
        msg = 'Skipping: NaN remains in exog columns after fill'
        print(f'  !! {msg}')
        status_rows.append({'commodity': commodity, 'status': 'skipped', 'reason': msg})
        continue

    try:
        print(f'  Fitting {MODEL_LABEL} with {len(exog_cols)} exog vars ...')
        model = SARIMAX(
            endog=y_train,
            exog=x_train,
            order=(P, D, Q),
            seasonal_order=(0, 0, 0, 0),
            trend='c',
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False)
        print(f'  AIC={fit.aic:.2f}  BIC={fit.bic:.2f}')

        # Forecast
        fc   = fit.get_forecast(steps=len(test_df), exog=x_test)
        pred_mean = np.asarray(fc.predicted_mean, dtype=float)
        ci        = fc.conf_int(alpha=0.05)
        ci80      = fc.conf_int(alpha=0.20)

        pred_df_out = pd.DataFrame({
            'mean':      pred_mean,
            'lower_95':  ci.iloc[:, 0].values,
            'upper_95':  ci.iloc[:, 1].values,
            'lower_80':  ci80.iloc[:, 0].values,
            'upper_80':  ci80.iloc[:, 1].values,
        })

        # Metrics (use original y_test values, not reset)
        m = compute_metrics(
            y_test.values, pred_mean, y_train.values,
            fit, commodity, MODEL_LABEL, len(exog_cols)
        )
        metrics_rows.append(m)
        print(f'  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  MASE={m["MASE"]:.4f}  '
              f'sMAPE={m["sMAPE"]:.4f}  MAPE={m["MAPE"]:.4f}')

        # Residual diagnostics
        d = residual_diagnostics(fit, commodity, MODEL_LABEL)
        diag_rows.append(d)
        print(f'  Ljung-Box(10) p={d["ljung_box_lag10_p"]:.4f}  '
              f'Normality p={d["normality_p"]:.4f}')

        # Coefficients
        c_rows = extract_coefficients(fit, commodity, MODEL_LABEL)
        coef_rows.extend(c_rows)

        # Save per-commodity forecast CSV
        fc_out = pd.DataFrame({
            'date':      pd.to_datetime(test_df['date']).dt.date,
            'commodity': commodity,
            'actual':    y_test.values,
            'predicted': pred_mean,
            'residual':  y_test.values - pred_mean,
            'lower_80':  pred_df_out['lower_80'].values,
            'upper_80':  pred_df_out['upper_80'].values,
            'lower_95':  pred_df_out['lower_95'].values,
            'upper_95':  pred_df_out['upper_95'].values,
            'model':     MODEL_LABEL,
        })
        fc_out.to_csv(FORECAST_DIR / f'{safe}.csv', index=False)

        # Plots
        plot_forecast(train_df, test_df, pred_df_out, commodity, safe, PLOT_DIR)
        plot_residuals(fit, commodity, safe, PLOT_DIR)
        plot_coefficients(c_rows, commodity, safe, PLOT_DIR)

        status_rows.append({'commodity': commodity, 'status': 'ok', 'reason': ''})

    except Exception as exc:
        print(f'  !! FAILED: {exc}')
        status_rows.append({'commodity': commodity, 'status': 'failed', 'reason': str(exc)})

print('\n\nLoop complete.')


In [ ]:
# ── Save summary CSVs ──────────────────────────────────────────────────────
metrics_df = pd.DataFrame(metrics_rows).sort_values('commodity').reset_index(drop=True)
diag_df    = pd.DataFrame(diag_rows).sort_values('commodity').reset_index(drop=True)
coef_df    = pd.DataFrame(coef_rows).sort_values(['commodity','term']).reset_index(drop=True)
status_df  = pd.DataFrame(status_rows).sort_values('commodity').reset_index(drop=True)

metrics_df.to_csv(OUTPUT_DIR / 'arimax_metrics.csv', index=False)
diag_df.to_csv(OUTPUT_DIR / 'arimax_residual_diagnostics.csv', index=False)
coef_df.to_csv(OUTPUT_DIR / 'arimax_coefficients.csv', index=False)
status_df.to_csv(OUTPUT_DIR / 'arimax_run_status.csv', index=False)

print('✅  Saved:')
print(f'   {OUTPUT_DIR / "arimax_metrics.csv"}')
print(f'   {OUTPUT_DIR / "arimax_residual_diagnostics.csv"}')
print(f'   {OUTPUT_DIR / "arimax_coefficients.csv"}')
print(f'   {OUTPUT_DIR / "arimax_run_status.csv"}')
print(f'   {FORECAST_DIR}/<commodity>.csv  (one per commodity)')
print(f'   {PLOT_DIR}/<commodity>_forecast/residuals/coefs.png')

In [ ]:
# ── Summary tables ─────────────────────────────────────────────────────────
print('\n--- ARIMAX Metrics ---')
display(metrics_df[['commodity','model','RMSE','MAE','MASE','sMAPE','MAPE','AIC','BIC','n_exog']])

print('\n--- Best commodity by RMSE ---')
display(metrics_df.sort_values('RMSE').head(5)[['commodity','RMSE','MAE','MASE']])

print('\n--- Residual Diagnostics ---')
display(diag_df[['commodity','resid_mean','resid_sd',
                 'ljung_box_lag10_p','normality_p','autocorr_ok','normality_ok']])

print('\n--- Significant Exog Coefficients (across all commodities) ---')
sig_coefs = coef_df[
    coef_df['significant'] &
    ~coef_df['term'].str.startswith(('ar.', 'ma.', 'sigma', 'const', 'intercept'))
].sort_values(['commodity','p_value'])
display(sig_coefs[['commodity','term','estimate','std_err','p_value','conf_lo','conf_hi']])

print('\n--- Run Status ---')
display(status_df)